# FieldCare v2 — standalone private-Kaggle training

This notebook is self-contained: it does not clone, import, or call anything from another repository.

Run all cells on a Jupyter PyTorch GPU instance. It authenticates with Kaggle, downloads and validates the private dataset, trains ImageNet-pretrained EfficientNet-B3, evaluates the untouched test split, saves models and detailed metrics, generates confusion matrices and training curves, and creates one downloadable artifact bundle.

## 1. Environment and output folders

The provider's PyTorch installation is retained so its CUDA build remains compatible with the GPU. Only small missing utilities are installed.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
import csv, getpass, hashlib, importlib.util, json, logging
import math, os, random, re, shutil, subprocess, sys, tarfile, time, zipfile

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Use a Jupyter PyTorch GPU template; PyTorch/torchvision is missing.")

packages = {"PIL":"pillow", "numpy":"numpy", "sklearn":"scikit-learn",
            "tqdm":"tqdm", "kagglehub":"kagglehub", "matplotlib":"matplotlib"}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.io import ImageReadMode, decode_jpeg, read_file
from torchvision.models import EfficientNet_B3_Weights, efficientnet_b3
from tqdm.auto import tqdm
import torchvision

BASE = (Path.cwd() / "fieldcare_training").resolve()
DOWNLOAD_DIR, DATA_DIR = BASE/"downloads", BASE/"data"
DATASET_ROOT, ARCHIVE = DATA_DIR/"dataset", DOWNLOAD_DIR/"dataset.tar"
MODEL_ROOT, OUTPUT_ROOT, LOG_ROOT, ARTIFACT_ROOT = (
    BASE/"models", BASE/"outputs", BASE/"logs", BASE/"artifacts"
)
for directory in (DOWNLOAD_DIR, DATA_DIR, MODEL_ROOT, OUTPUT_ROOT, LOG_ROOT, ARTIFACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "priyanshudebnath348/fieldcare-plant-disease-dataset-v2"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DOWNLOAD_LOG = LOG_ROOT/"kaggle_dataset"/f"{STAMP}_download.log"
DOWNLOAD_LOG.parent.mkdir(parents=True, exist_ok=True)

def event(message):
    with DOWNLOAD_LOG.open("a", encoding="utf-8") as handle:
        handle.write(f"{datetime.now().isoformat(timespec='seconds')} | {message}\n")

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__,
      "| torchvision:", torchvision.__version__, "| CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU instance with the Jupyter PyTorch template.")
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
try:
    probe = torch.ones(1, device=DEVICE) * 2
    torch.cuda.synchronize()
    del probe
except RuntimeError as exc:
    raise RuntimeError("The provider PyTorch build does not support this GPU architecture.") from exc
print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB:.1f} GiB | workspace: {BASE}")
event(f"Started; torch={torch.__version__}; gpu={GPU_NAME}; vram={GPU_VRAM_GB:.1f}GiB")

## 2. Kaggle authentication

Authentication uses Kaggle's supported `kagglehub` login. If no valid session exists, paste the new API token from Kaggle Settings → API → Generate New Token into the hidden prompt. A username is not requested, and the token is not stored in this notebook.

In [ ]:
import kagglehub

owner = DATASET_SLUG.split("/", 1)[0]
try:
    identity = kagglehub.whoami()
except Exception:
    print("Paste the NEW Kaggle API token (not the legacy username/key pair).")
    kagglehub.login()
    identity = kagglehub.whoami()

authenticated_user = identity.get("username", "")
if authenticated_user.casefold() != owner.casefold():
    raise RuntimeError(
        f"Authenticated as {authenticated_user!r}; private dataset owner is {owner!r}."
    )
print(f"KaggleHub authenticated as: {authenticated_user}")
event(f"KaggleHub authentication accepted for {authenticated_user}")

## 3. Download and safely extract

Kaggle unpacked the uploaded archive during dataset processing, so the remote resource contains `dataset/...` files rather than a downloadable archive. This cell downloads the complete dataset directly into the local data directory and reuses it on later runs.

In [ ]:
def metadata_ready():
    return (DATASET_ROOT/"labels.json").is_file() and (DATASET_ROOT/"manifest.csv").is_file()

if metadata_ready():
    print("Reusing downloaded dataset:", DATASET_ROOT)
else:
    free = shutil.disk_usage(BASE).free / 1024**3
    print(f"Free disk: {free:.1f} GiB")
    if free < 12:
        raise RuntimeError("At least 12 GiB free is required to download this dataset.")
    print("Downloading the complete Kaggle dataset (about 9.1 GiB)...")
    event("Starting complete dataset download with KaggleHub")
    downloaded_path = Path(kagglehub.dataset_download(
        DATASET_SLUG, output_dir=str(DATA_DIR)
    ))
    print("KaggleHub returned:", downloaded_path)
    if not metadata_ready():
        raise RuntimeError(
            f"Download completed, but labels.json/manifest.csv was not found under {DATASET_ROOT}. "
            f"KaggleHub returned {downloaded_path}."
        )
    event(f"Complete dataset downloaded to {DATASET_ROOT}")

print("Dataset ready:", DATASET_ROOT)

## 4. Dataset model and leakage audit

The manifest is the source of truth. The audit checks every referenced path, all class/split combinations, contiguous label indices, and augmentation-family separation across train, validation, and test.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

VALID_EXTENSIONS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
SPLITS = ("train","val","test")

@dataclass(frozen=True)
class ClassInfo:
    id: int
    index: int
    key: str
    display_name: str
    crop: str
    disease: str
    relative_directory: str

@dataclass(frozen=True)
class Sample:
    path: Path
    class_index: int
    family_id: str

class LeafDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        sample = self.samples[index]
        try:
            with Image.open(sample.path) as image:
                return self.transform(image.convert("RGB")), sample.class_index
        except Exception as exc:
            raise RuntimeError(f"Unreadable image: {sample.path}") from exc

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

labels = json.loads((DATASET_ROOT/"labels.json").read_text(encoding="utf-8"))
CLASSES = [ClassInfo(**item) for item in labels.get("classes", [])]
if not CLASSES:
    raise RuntimeError("No classes in labels.json")
if [c.id for c in CLASSES] != list(range(1, len(CLASSES)+1)):
    raise RuntimeError("Class IDs are not contiguous from 1.")
if [c.index for c in CLASSES] != list(range(len(CLASSES))):
    raise RuntimeError("Class indices are not contiguous from 0.")

SAMPLES = {split:[] for split in SPLITS}
counts, families = defaultdict(Counter), defaultdict(set)
manifest_path = DATASET_ROOT/"manifest.csv"
with manifest_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    required = {"split","class_id","class_index","class_key","family_id","relative_path"}
    if not required.issubset(reader.fieldnames or []):
        raise RuntimeError(f"Manifest columns missing: {sorted(required-set(reader.fieldnames or []))}")
    by_id = {c.id:c for c in CLASSES}
    for line, row in enumerate(reader, 2):
        split, class_id, class_index = row["split"], int(row["class_id"]), int(row["class_index"])
        if split not in SAMPLES:
            raise RuntimeError(f"Invalid split at line {line}: {split}")
        info = by_id.get(class_id)
        if info is None or info.index != class_index or info.key != row["class_key"]:
            raise RuntimeError(f"Class mismatch at manifest line {line}")
        path = DATASET_ROOT/Path(row["relative_path"])
        if path.suffix.lower() not in VALID_EXTENSIONS:
            raise RuntimeError(f"Unsupported file: {path}")
        family = row["family_id"]
        SAMPLES[split].append(Sample(path, class_index, family))
        counts[info.key][split] += 1
        families[(class_id, family)].add(split)

all_paths = [sample.path for split in SPLITS for sample in SAMPLES[split]]
audit_threads = min(32, max(4, (os.cpu_count() or 4)*2))
print(f"Verifying {len(all_paths):,} image paths with {audit_threads} threads...")
def missing_in_chunk(paths):
    return [path for path in paths if not path.is_file()]
chunks = [all_paths[start:start+4096] for start in range(0, len(all_paths), 4096)]
missing = []
with ThreadPoolExecutor(max_workers=audit_threads) as executor:
    for absent in executor.map(missing_in_chunk, chunks):
        missing.extend(absent)
if missing:
    raise RuntimeError(f"{len(missing)} missing images; examples: {[str(path) for path in missing[:20]]}")
leakage = [key for key, assigned in families.items() if len(assigned) > 1]
if leakage:
    raise RuntimeError(f"Augmentation-family leakage: {len(leakage)} groups")
for info in CLASSES:
    for split in SPLITS:
        if not counts[info.key][split]:
            raise RuntimeError(f"{info.display_name} has no {split} images")

DATASET_SUMMARY = {
    "manifest_sha256": sha256(manifest_path),
    "total_images": sum(map(len, SAMPLES.values())),
    "split_counts": {split:len(SAMPLES[split]) for split in SPLITS},
    "family_count": len(families),
    "family_leakage_count": len(leakage),
    "per_class_split": {c.key:{s:counts[c.key][s] for s in SPLITS} for c in CLASSES},
}
print(f"Classes: {len(CLASSES)} | images: {DATASET_SUMMARY['total_images']:,} | "
      f"families: {len(families):,} | leakage: {len(leakage)}")
print("Splits:", DATASET_SUMMARY["split_counts"])
for c in CLASSES:
    n = DATASET_SUMMARY["per_class_split"][c.key]
    print(f"{c.id:2d} {c.display_name:<47} {n['train']:>6,}/{n['val']:>6,}/{n['test']:>6,}")
event(f"Audit passed: {DATASET_SUMMARY['total_images']} images, {len(CLASSES)} classes")

## 5. Training configuration

The dataset already contains extensive offline transformations, so online augmentation remains moderate. Validation and test preprocessing is deterministic. Batch size is selected from GPU VRAM; reduce it if CUDA reports out-of-memory.

In [ ]:
SEED, EPOCHS, WARMUP_EPOCHS, IMAGE_SIZE = 42, 20, 2, 300
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
DROPOUT, LABEL_SMOOTHING, BALANCE_BETA, PATIENCE = 0.4, 0.1, 0.9999, 6
USE_AMP = True
BATCH_SIZE = 192 if GPU_VRAM_GB >= 80 else 128 if GPU_VRAM_GB >= 40 else 64 if GPU_VRAM_GB >= 28 else 48 if GPU_VRAM_GB >= 20 else 32 if GPU_VRAM_GB >= 12 else 16
DESIRED_WORKERS = min(14, max(2, (os.cpu_count() or 4)-2))
PREFETCH_FACTOR = 2
SHM_GB = shutil.disk_usage("/dev/shm").total/1024**3 if Path("/dev/shm").exists() else 0.0
batch_bytes = BATCH_SIZE*3*IMAGE_SIZE*IMAGE_SIZE*4
safe_prefetched_batches = int((SHM_GB*1024**3*0.70)//batch_bytes) if SHM_GB else 0
WORKERS = min(DESIRED_WORKERS, safe_prefetched_batches//PREFETCH_FACTOR)
if WORKERS < 1:
    WORKERS = 0
    print(f"Shared memory is only {SHM_GB:.2f} GiB; using WORKERS=0 to avoid DataLoader crashes.")
else:
    print(f"Shared memory: {SHM_GB:.2f} GiB; using {WORKERS} DataLoader workers.")
RUN_NAME = f"fieldcare_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
MODEL_DIR, OUTPUT_DIR, TRAIN_LOG_DIR = MODEL_ROOT/RUN_NAME, OUTPUT_ROOT/RUN_NAME, LOG_ROOT/"training"/RUN_NAME
for directory in (MODEL_DIR, OUTPUT_DIR, TRAIN_LOG_DIR):
    directory.mkdir(parents=True, exist_ok=False)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

normalize = transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.1),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=.12, contrast=.12, saturation=.12, hue=.02),
    transforms.ToTensor(), normalize,
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.ToTensor(), normalize,
])

set_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

def loader(split):
    generator = torch.Generator().manual_seed(SEED)
    kwargs = dict(dataset=LeafDataset(SAMPLES[split], train_transform if split=="train" else eval_transform),
                  batch_size=BATCH_SIZE, shuffle=split=="train", num_workers=WORKERS,
                  pin_memory=True, persistent_workers=WORKERS>0,
                  worker_init_fn=seed_worker, generator=generator)
    if WORKERS: kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

class CudaJpegLoader:
    """Read compressed bytes with threads; decode and preprocess batches on CUDA."""
    def __init__(self, samples, batch_size, training, io_threads, seed):
        self.samples = samples
        self.batch_size = batch_size
        self.training = training
        self.io_threads = io_threads
        self.seed = seed
        self.epoch = 0

    def __len__(self):
        return math.ceil(len(self.samples)/self.batch_size)

    def _preprocess(self, decoded):
        batch = torch.empty(
            (len(decoded), 3, IMAGE_SIZE, IMAGE_SIZE),
            device=DEVICE, dtype=torch.float32,
        )
        groups = defaultdict(list)
        for index, image in enumerate(decoded):
            groups[tuple(image.shape[-2:])].append(index)
        for group_indices in groups.values():
            images = torch.stack([decoded[index] for index in group_indices]).float()
            images = F.interpolate(
                images, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear",
                align_corners=False, antialias=True,
            ).div_(255.0)
            batch[torch.tensor(group_indices, device=DEVICE)] = images

        if self.training:
            horizontal = torch.rand(len(batch), device=DEVICE) < 0.5
            vertical = torch.rand(len(batch), device=DEVICE) < 0.1
            batch[horizontal] = batch[horizontal].flip(-1)
            batch[vertical] = batch[vertical].flip(-2)
            brightness = torch.empty((len(batch),1,1,1), device=DEVICE).uniform_(0.88,1.12)
            contrast = torch.empty((len(batch),1,1,1), device=DEVICE).uniform_(0.88,1.12)
            saturation = torch.empty((len(batch),1,1,1), device=DEVICE).uniform_(0.88,1.12)
            batch.mul_(brightness)
            spatial_mean = batch.mean(dim=(-2,-1), keepdim=True)
            batch = (batch-spatial_mean)*contrast+spatial_mean
            gray = (batch*torch.tensor([0.299,0.587,0.114], device=DEVICE).view(1,3,1,1)).sum(1,keepdim=True)
            batch = (batch-gray)*saturation+gray
            batch.clamp_(0,1)

        mean = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
        std = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)
        return ((batch-mean)/std).contiguous(memory_format=torch.channels_last)

    def __iter__(self):
        indices = np.arange(len(self.samples))
        if self.training:
            np.random.default_rng(self.seed+self.epoch).shuffle(indices)
            self.epoch += 1
        batches = [indices[start:start+self.batch_size] for start in range(0,len(indices),self.batch_size)]
        with ThreadPoolExecutor(max_workers=self.io_threads) as pool:
            pending = [pool.submit(read_file, str(self.samples[index].path)) for index in batches[0]]
            for batch_number, batch_indices in enumerate(batches):
                encoded = [future.result() for future in pending]
                if batch_number+1 < len(batches):
                    pending = [pool.submit(read_file, str(self.samples[index].path))
                               for index in batches[batch_number+1]]
                decoded = decode_jpeg(encoded, mode=ImageReadMode.RGB, device=DEVICE)
                images = self._preprocess(decoded)
                targets = torch.tensor(
                    [self.samples[index].class_index for index in batch_indices],
                    dtype=torch.long, device=DEVICE,
                )
                yield images, targets

IO_THREADS = min(14, max(4, (os.cpu_count() or 4)-2))
try:
    probe_bytes = read_file(str(SAMPLES["train"][0].path))
    probe_image = decode_jpeg([probe_bytes], mode=ImageReadMode.RGB, device=DEVICE)
    del probe_bytes, probe_image
    torch.cuda.empty_cache()
    USE_CUDA_JPEG = True
except Exception as exc:
    USE_CUDA_JPEG = False
    print(f"CUDA nvJPEG unavailable ({exc}); using the safe CPU loader.")

if USE_CUDA_JPEG:
    LOADER_BACKEND = "cuda_nvjpeg"
    LOADERS = {split:CudaJpegLoader(
        SAMPLES[split], BATCH_SIZE, split=="train", IO_THREADS, SEED
    ) for split in SPLITS}
    print(f"Using batched CUDA nvJPEG decode with {IO_THREADS} I/O threads.")
else:
    LOADER_BACKEND = "pillow_cpu"
    LOADERS = {split:loader(split) for split in SPLITS}
class_counts = np.bincount([s.class_index for s in SAMPLES["train"]], minlength=len(CLASSES)).astype(float)
effective = 1-np.power(BALANCE_BETA, class_counts)
class_weights = (1-BALANCE_BETA)/effective
class_weights = torch.tensor(class_weights/class_weights.sum()*len(CLASSES), dtype=torch.float32)

CONFIG = {
    "run_name":RUN_NAME, "dataset_slug":DATASET_SLUG, "architecture":"efficientnet_b3",
    "pretrained":True, "gpu":GPU_NAME, "gpu_vram_gib":GPU_VRAM_GB,
    "epochs":EPOCHS, "warmup_epochs":WARMUP_EPOCHS, "image_size":IMAGE_SIZE,
    "batch_size":BATCH_SIZE, "workers":WORKERS, "shared_memory_gib":SHM_GB,
    "prefetch_factor":PREFETCH_FACTOR if WORKERS else None, "loader_backend":LOADER_BACKEND,
    "io_threads":IO_THREADS if USE_CUDA_JPEG else None, "head_lr":HEAD_LR,
    "backbone_lr":BACKBONE_LR, "weight_decay":WEIGHT_DECAY, "dropout":DROPOUT,
    "label_smoothing":LABEL_SMOOTHING, "class_balance_beta":BALANCE_BETA,
    "early_stop_patience":PATIENCE, "seed":SEED, "amp":USE_AMP,
    "dataset_summary":DATASET_SUMMARY, "classes":[asdict(c) for c in CLASSES],
}
(OUTPUT_DIR/"config.json").write_text(json.dumps(CONFIG, indent=2)+"\n", encoding="utf-8")
print(f"Run: {RUN_NAME} | epochs: {EPOCHS} | batch: {BATCH_SIZE} | loader: {LOADER_BACKEND}")

## 6. Training and evaluation functions

Checkpoint selection uses validation macro-F1. The test split remains untouched until training and early stopping have completed.

In [ ]:
def setup_logger():
    logger = logging.getLogger(RUN_NAME); logger.setLevel(logging.INFO); logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    for handler in (logging.StreamHandler(sys.stdout),
                    logging.FileHandler(TRAIN_LOG_DIR/"training.log", encoding="utf-8")):
        handler.setFormatter(formatter); logger.addHandler(handler)
    return logger

def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train(); total_loss=correct=count=0
    progress = tqdm(loader, desc="Training", leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits=model(images); loss=criterion(logits, targets)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        n=targets.size(0); total_loss += loss.item()*n
        correct += (logits.argmax(1)==targets).sum().item(); count += n
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return {"loss":total_loss/count, "accuracy":correct/count}

def evaluate(model, loader, criterion, description):
    model.eval(); total_loss=count=0; predictions=[]; targets_all=[]; confidences=[]
    with torch.inference_mode():
        for images, targets in tqdm(loader, desc=description, leave=False):
            images = images.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits=model(images); loss=criterion(logits, targets); probs=torch.softmax(logits,1)
            confidence, prediction = probs.max(1); n=targets.size(0)
            total_loss += loss.item()*n; count += n
            predictions.extend(prediction.cpu().tolist())
            targets_all.extend(targets.cpu().tolist()); confidences.extend(confidence.cpu().tolist())
    y, p = np.asarray(targets_all), np.asarray(predictions)
    return {"loss":total_loss/count, "accuracy":float((y==p).mean()),
            "macro_f1":float(f1_score(y,p,average="macro",zero_division=0)),
            "weighted_f1":float(f1_score(y,p,average="weighted",zero_division=0)),
            "balanced_accuracy":float(balanced_accuracy_score(y,p)),
            "predictions":predictions, "targets":targets_all, "confidences":confidences}

def compact(metrics):
    return {k:v for k,v in metrics.items() if k not in {"predictions","targets","confidences"}}

def checkpoint(model, epoch, validation, test=None):
    result = {"model_state_dict":model.state_dict(), "architecture":"efficientnet_b3",
              "num_classes":len(CLASSES), "classes":[asdict(c) for c in CLASSES],
              "labels":{c.id:c.display_name for c in CLASSES}, "image_size":IMAGE_SIZE,
              "normalization":{"mean":[.485,.456,.406],"std":[.229,.224,.225]},
              "epoch":epoch, "validation_metrics":validation,
              "manifest_sha256":DATASET_SUMMARY["manifest_sha256"], "seed":SEED, "config":CONFIG}
    if test is not None: result["test_metrics"] = test
    return result

LOGGER = setup_logger()

## 7. Start paid GPU training

The best model is saved after every macro-F1 improvement. Interrupting the cell also writes an interruption checkpoint.

In [ ]:
MODEL = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
features = MODEL.classifier[1].in_features
MODEL.classifier = nn.Sequential(nn.Dropout(DROPOUT, inplace=True), nn.Linear(features,len(CLASSES)))
MODEL = MODEL.to(DEVICE).to(memory_format=torch.channels_last)
CRITERION = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=LABEL_SMOOTHING)
try: SCALER = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError: SCALER = torch.cuda.amp.GradScaler(enabled=USE_AMP)

BEST_PATH = MODEL_DIR/"fieldcare_v2_efficientnet_b3_best.pth"
FINAL_PATH = MODEL_DIR/"fieldcare_v2_efficientnet_b3_final.pth"
INTERRUPTED_PATH = MODEL_DIR/"fieldcare_v2_efficientnet_b3_interrupted.pth"
for parameter in MODEL.features.parameters(): parameter.requires_grad = WARMUP_EPOCHS==0
optimizer = torch.optim.AdamW(MODEL.classifier.parameters(),lr=HEAD_LR,weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,WARMUP_EPOCHS))
HISTORY=[]; best_f1=-math.inf; best_epoch=0; stale=0; started=time.time()
LOGGER.info("Starting %s on %s; batch=%d", RUN_NAME, GPU_NAME, BATCH_SIZE)

try:
    for epoch in range(1,EPOCHS+1):
        if epoch==WARMUP_EPOCHS+1 and WARMUP_EPOCHS:
            for parameter in MODEL.features.parameters(): parameter.requires_grad=True
            optimizer=torch.optim.AdamW([
                {"params":MODEL.features.parameters(),"lr":BACKBONE_LR},
                {"params":MODEL.classifier.parameters(),"lr":HEAD_LR}],weight_decay=WEIGHT_DECAY)
            scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,T_max=max(1,EPOCHS-WARMUP_EPOCHS))
            LOGGER.info("Backbone unfrozen")
        phase="warmup" if epoch<=WARMUP_EPOCHS else "fine_tune"
        LOGGER.info("Epoch %d/%d (%s)",epoch,EPOCHS,phase)
        train=train_epoch(MODEL,LOADERS["train"],CRITERION,optimizer,SCALER)
        val=evaluate(MODEL,LOADERS["val"],CRITERION,"Validation"); scheduler.step()
        row={"epoch":epoch,"phase":phase,"train_loss":train["loss"],
             "train_accuracy":train["accuracy"],"val_loss":val["loss"],
             "val_accuracy":val["accuracy"],"val_macro_f1":val["macro_f1"],
             "val_weighted_f1":val["weighted_f1"],
             "val_balanced_accuracy":val["balanced_accuracy"],
             "learning_rates":[g["lr"] for g in optimizer.param_groups]}
        HISTORY.append(row)
        (OUTPUT_DIR/"history.json").write_text(json.dumps(HISTORY,indent=2)+"\n",encoding="utf-8")
        LOGGER.info("train loss=%.4f acc=%.2f%% | val loss=%.4f acc=%.2f%% macro_f1=%.4f",
                    train["loss"],train["accuracy"]*100,val["loss"],val["accuracy"]*100,val["macro_f1"])
        if val["macro_f1"]>best_f1:
            best_f1,best_epoch,stale=val["macro_f1"],epoch,0
            torch.save(checkpoint(MODEL,epoch,compact(val)),BEST_PATH)
            LOGGER.info("Saved best checkpoint")
        else:
            stale += 1
            if stale>=PATIENCE:
                LOGGER.info("Early stopping after %d stale epochs",stale); break
except KeyboardInterrupt:
    torch.save(checkpoint(MODEL,HISTORY[-1]["epoch"] if HISTORY else 0,{}),INTERRUPTED_PATH)
    LOGGER.warning("Interrupted; saved %s",INTERRUPTED_PATH)
    raise

best = torch.load(BEST_PATH,map_location=DEVICE,weights_only=False)
MODEL.load_state_dict(best["model_state_dict"])
TEST_RESULTS=evaluate(MODEL,LOADERS["test"],CRITERION,"Held-out test")
TEST_METRICS=compact(TEST_RESULTS); elapsed=time.time()-started
torch.save(checkpoint(MODEL,best["epoch"],best["validation_metrics"],TEST_METRICS),FINAL_PATH)
SUMMARY={"run_name":RUN_NAME,"best_epoch":best_epoch,"best_validation_macro_f1":best_f1,
         "test_metrics":TEST_METRICS,"epochs_completed":len(HISTORY),
         "training_seconds":elapsed,"best_checkpoint":str(BEST_PATH),
         "final_checkpoint":str(FINAL_PATH),"manifest_sha256":DATASET_SUMMARY["manifest_sha256"]}
(OUTPUT_DIR/"summary.json").write_text(json.dumps(SUMMARY,indent=2)+"\n",encoding="utf-8")
LOGGER.info("Test accuracy=%.2f%% macro_f1=%.4f balanced_accuracy=%.2f%%",
            TEST_METRICS["accuracy"]*100,TEST_METRICS["macro_f1"],
            TEST_METRICS["balanced_accuracy"]*100)
print(f"Best epoch: {best_epoch} | val macro-F1: {best_f1:.4f}")
print(f"Test accuracy: {TEST_METRICS['accuracy']*100:.2f}% | macro-F1: {TEST_METRICS['macro_f1']:.4f} | "
      f"balanced accuracy: {TEST_METRICS['balanced_accuracy']*100:.2f}%")

## 8. Reports, confusion matrices, curves, and portable bundle

Download the final TAR.GZ bundle from Jupyter before deleting the temporary GPU instance.

In [ ]:
names=[c.display_name for c in CLASSES]; indices=list(range(len(CLASSES)))
targets,predictions,confidences=(TEST_RESULTS["targets"],TEST_RESULTS["predictions"],TEST_RESULTS["confidences"])
report=classification_report(targets,predictions,labels=indices,target_names=names,
                             output_dict=True,zero_division=0)
report_text=classification_report(targets,predictions,labels=indices,target_names=names,zero_division=0)
matrix=confusion_matrix(targets,predictions,labels=indices)
normalized=np.divide(matrix,matrix.sum(1,keepdims=True),out=np.zeros_like(matrix,dtype=float),
                     where=matrix.sum(1,keepdims=True)!=0)
(OUTPUT_DIR/"classification_report.json").write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
(OUTPUT_DIR/"classification_report.txt").write_text(report_text+"\n",encoding="utf-8")
np.save(OUTPUT_DIR/"confusion_matrix.npy",matrix)
np.save(OUTPUT_DIR/"confusion_matrix_normalized.npy",normalized)

with (OUTPUT_DIR/"confusion_matrix.csv").open("w",newline="",encoding="utf-8") as handle:
    writer=csv.writer(handle); writer.writerow(["actual/predicted",*names])
    for name,row in zip(names,matrix): writer.writerow([name,*row.tolist()])
with (OUTPUT_DIR/"classification_report.csv").open("w",newline="",encoding="utf-8") as handle:
    writer=csv.writer(handle); writer.writerow(["class","precision","recall","f1-score","support"])
    for name,values in report.items():
        if isinstance(values,dict):
            writer.writerow([name,values.get("precision",""),values.get("recall",""),
                             values.get("f1-score",""),values.get("support","")])
with (OUTPUT_DIR/"predictions.csv").open("w",newline="",encoding="utf-8") as handle:
    fields=["image","actual_class_id","actual_name","predicted_class_id",
            "predicted_name","confidence","correct"]
    writer=csv.DictWriter(handle,fieldnames=fields); writer.writeheader()
    for sample,actual,predicted,confidence in zip(SAMPLES["test"],targets,predictions,confidences):
        writer.writerow({"image":sample.path.relative_to(DATASET_ROOT).as_posix(),
            "actual_class_id":CLASSES[actual].id,"actual_name":names[actual],
            "predicted_class_id":CLASSES[predicted].id,"predicted_name":names[predicted],
            "confidence":round(confidence,7),"correct":actual==predicted})
with (OUTPUT_DIR/"history.csv").open("w",newline="",encoding="utf-8") as handle:
    writer=csv.DictWriter(handle,fieldnames=list(HISTORY[0])); writer.writeheader(); writer.writerows(HISTORY)

fig,axes=plt.subplots(1,2,figsize=(24,10))
im0=axes[0].imshow(np.log1p(matrix),cmap="magma"); fig.colorbar(im0,ax=axes[0],fraction=.046)
im1=axes[1].imshow(normalized,cmap="Blues",vmin=0,vmax=1); fig.colorbar(im1,ax=axes[1],fraction=.046)
axes[0].set_title("Test confusion matrix — log(1 + count)")
axes[1].set_title("Test confusion matrix — row normalized")
for axis in axes:
    axis.set_xticks(indices); axis.set_yticks(indices)
    axis.set_xticklabels(names,rotation=90,fontsize=7); axis.set_yticklabels(names,fontsize=7)
    axis.set_xlabel("Predicted"); axis.set_ylabel("Actual")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"confusion_matrix.png",dpi=180,bbox_inches="tight")
plt.show(); plt.close(fig)

epochs=[r["epoch"] for r in HISTORY]; fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].plot(epochs,[r["train_loss"] for r in HISTORY],label="train")
axes[0].plot(epochs,[r["val_loss"] for r in HISTORY],label="validation"); axes[0].set_title("Loss")
axes[1].plot(epochs,[r["train_accuracy"]*100 for r in HISTORY],label="train")
axes[1].plot(epochs,[r["val_accuracy"]*100 for r in HISTORY],label="validation"); axes[1].set_title("Accuracy (%)")
axes[2].plot(epochs,[r["val_macro_f1"] for r in HISTORY],label="macro-F1")
axes[2].plot(epochs,[r["val_balanced_accuracy"] for r in HISTORY],label="balanced accuracy")
axes[2].set_title("Validation balanced metrics")
for axis in axes: axis.set_xlabel("Epoch"); axis.grid(alpha=.25); axis.legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"training_curves.png",dpi=180,bbox_inches="tight")
plt.show(); plt.close(fig)

BUNDLE=ARTIFACT_ROOT/f"{RUN_NAME}.tar.gz"
with tarfile.open(BUNDLE,"w:gz") as archive:
    archive.add(MODEL_DIR,arcname=f"{RUN_NAME}/models")
    archive.add(OUTPUT_DIR,arcname=f"{RUN_NAME}/outputs")
    archive.add(TRAIN_LOG_DIR,arcname=f"{RUN_NAME}/logs/training")
    archive.add(DOWNLOAD_LOG,arcname=f"{RUN_NAME}/logs/{DOWNLOAD_LOG.name}")
print(report_text)
print("Best model :",BEST_PATH)
print("Final model:",FINAL_PATH)
print("Reports    :",OUTPUT_DIR)
print(f"DOWNLOAD THIS BUNDLE BEFORE DELETING THE SERVER:\n{BUNDLE}\nSize: {BUNDLE.stat().st_size/1024**2:.1f} MiB")